In [1]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns

from glob import glob # list out all files in a directory

import soundfile as sf
import librosa
import librosa.display

from pathlib import Path

from sklearn.model_selection import train_test_split

## 1. Setting directory

In [2]:
audio_files = glob("/Users/mayawiegand/Documents/ECS 171/Project/music-genre-classification/Raw Audio Data/*/*.wav") # creating a list of all of the audio files for all of the genres

## 2. Grabbing genre labels and file paths

In [3]:
# initializing lists to store genre labels and original file paths

genre_list = []
paths_list = []

for audio in audio_files:
    current_genre = Path(audio).parent.name # grabbing the folder name of the parent folder which is the genre label
    genre_list.append(current_genre) # adding this to genre list

    current_path = Path(audio) # grabbing the full file path (just in case we need later)
    paths_list.append(current_path) # adding this to file path list

## 3. Splitting data into training and testing set
- Need to split into training and testing before audio clips are split into smaller segments (to give us more training instances and hopefully improve model) to prevent data leakage
- Audio data typically uses y to represent the audio data - stayed consistent above with this
- Now that training and testing datasets are being built, stayed consistent with ML:
    - y = genre label
    - X = path to each of the audio files

In [4]:
# genre list/paths list for training/testing split so don't have to load all audio files, then reload to create spectrograms

X_train, X_test, y_train, y_test = train_test_split(paths_list, genre_list, test_size=0.2, random_state=42, stratify=genre_list)

# using stratify here so that the genre proportions are consistent 
# train/test split is done separately within each class to make sure each genre is represented proportionately in training and testing data

# 4. Grabbing a Classical Song from Testing Set
- Grabbing one classical song and saving it separately from the testing set to use in the demo


In [5]:
demo_index = y_test.index("classical") 
X_demo = [X_test[demo_index]]
y_demo = [y_test[demo_index]]
X_test.pop(demo_index)
y_test.pop(demo_index)

'classical'

## 4. Segmenting Audo Files, Re-Sampling, and Creating Spectrograms

In [6]:
def create_spectro(audio_list, label_list):
    
    spectro_list = [] # initializing a list to keep the spectrograms
    song_labels = [] # initializing a list to keep the genre labels (true label)
    song_paths = [] # initializing a list to keep the song paths (in case we later want to match the exact song to an observation)

    target_sr = 22050

    for audio, label in zip(audio_list, label_list):

        y, sr = sf.read(audio) # using sound file to read in audio, y = raw audio data and sr = sampling rate (how often the audio is sampled by the computer since it isn't continuous like human ears hear it)
        
        # converting to one audio channel (need one-dimensional to create the spectrogram)
        if y.ndim == 2:
            y = y.mean(axis=1)
        
        y = y.astype(float) # need y to be a float when computing spectrogram

        if sr != target_sr:
            y = librosa.resample(y, orig_sr=sr, target_sr=target_sr) # resampling to 22050 to make sure all files have consistent sampling rate
            sr = target_sr

        y, _ = librosa.effects.trim(y) # trimming any silence from the beginning/end of tracks
        
        y = librosa.util.fix_length(y, size=target_sr * 30) # clipping songs to ensure each is exactly 30 seconds (need all time dimensions to match when feeding into models)

        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128) # creating mel spectrogram, n_mels = how many perceptual frequency bands do you want (how finely to slice frequency axis to best represent how humans hear it)
        S_db_mel = librosa.power_to_db(S, ref=np.max) # converting to log decibels (so this can be understood as volume)
            
        spectro_list.append(S_db_mel) # adding this spectrogram to the list
        song_labels.append(label) # adding this genre to the list
        song_paths.append(audio) # adding path to the list

    return spectro_list, song_labels, song_paths

X_train_spec, y_train_labels, train_paths = create_spectro(X_train, y_train)
X_test_spec, y_test_labels, test_paths = create_spectro(X_test, y_test)
X_demo_spec, y_demo_labels, demo_path = create_spectro(X_demo, y_demo)

In [7]:
# converting to arrays

X_train_spec = np.array(X_train_spec)
X_test_spec = np.array(X_test_spec)
X_demo_spec = np.array(X_demo_spec)

y_train = np.array(y_train_labels)
y_test = np.array(y_test_labels)
y_demo = np.array(y_demo_labels)

train_paths = np.array(train_paths)
test_paths = np.array(train_paths)
demo_path = np.array(demo_path)

## 4. Standardization

In [8]:
# using (x-mean of training)/sd of training to standardize both training and testing spectrograms

mu = X_train_spec.mean()
sigma = X_train_spec.std() + 1e-8 # adding 1e^-8 prevents the denominator from being 0 (possible result from numerical precision, and parts across a spectrogram can be constant which could lead to 0/very small variance)

X_train_spec = (X_train_spec - mu) / sigma
X_test_spec = (X_test_spec - mu) / sigma
X_demo_spec = (X_demo_spec - mu) / sigma

The final X_train, X_test and there corresponding genre labels X_train_labels, y_train_lables are ready for the model.

## 5. Saving Spectrograms, Lables, and Paths

In [9]:
save_dir = Path("processed_data")
save_dir.mkdir(exist_ok=True)

# save spectrogram arrays
np.save(save_dir / "X_train.npy", X_train_spec)
np.save(save_dir / "X_test.npy", X_test_spec)
np.save(save_dir / "X_demo.npy", X_demo_spec)

# save labels
np.save(save_dir / "y_train.npy", y_train)
np.save(save_dir / "y_test.npy", y_test)
np.save(save_dir / "y_demo.npy", y_demo)

# save file paths
np.save(save_dir / "training_file_paths.npy", train_paths)
np.save(save_dir / "testing_file_paths.npy", test_paths)
np.save(save_dir / "demo_file_path.npy", demo_path)

print(list(save_dir.glob("*")))

[PosixPath('processed_data/.DS_Store'), PosixPath('processed_data/y_demo.npy'), PosixPath('processed_data/X_demo.npy'), PosixPath('processed_data/y_train.npy'), PosixPath('processed_data/flattened_y_train.npy'), PosixPath('processed_data/testing_file_paths.npy'), PosixPath('processed_data/y_test.npy'), PosixPath('processed_data/X_test.npy'), PosixPath('processed_data/demo_file_path.npy'), PosixPath('processed_data/X_train.npy'), PosixPath('processed_data/flattened_X_train.npy'), PosixPath('processed_data/training_file_paths.npy'), PosixPath('processed_data/flattened_y_test.npy'), PosixPath('processed_data/flattened_X_test.npy')]
